# LeetCode #82 — Remove Duplicates from Sorted List II: xóa sạch mọi giá trị bị lặp

**Mức độ:** Medium · **Chủ đề:** Linked List, Two Pointers

Skeleton trên LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def deleteDuplicates(self, head: Optional[ListNode]) -> Optional[ListNode]:
```

Khác với #83 (giữ lại **một** bản), bài này xóa **toàn bộ** các nút có giá trị bị lặp. Mục tiêu của notebook:

1. Dùng **nút giả (dummy)** vì nút đầu cũng có thể bị xóa.
2. Con trỏ `prev` (nút cuối cùng **chắc chắn được giữ**) và `cur` (đang xét).
3. Bỏ qua **cả cụm** giá trị lặp bằng một vòng `while`.
4. So sánh với cách đệ quy.

# 1. Bài toán là gì?

Cho `head` của một danh sách liên kết **đã sắp xếp**. Xóa mọi nút có giá trị **xuất hiện nhiều hơn một lần**, chỉ giữ các giá trị **duy nhất**. Trả về danh sách (vẫn sắp xếp).

**Constraints (theo đề):**

- Số nút trong khoảng `[0, 300]`.
- `-100 <= Node.val <= 100`
- Danh sách được đảm bảo sắp xếp tăng dần (không giảm).

---

## Ví dụ

```text
1 -> 2 -> 3 -> 3 -> 4 -> 4 -> 5   ->   1 -> 2 -> 5
1 -> 1 -> 1 -> 2 -> 3             ->   2 -> 3
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Danh sách liên kết đơn (singly linked list)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Danh sách liên kết đơn (singly linked list)

### Danh sách liên kết là gì?

Mảng (`list` của Python) lưu các phần tử **liền nhau** trong bộ nhớ, nên truy cập `a[i]` tức thì. Danh sách liên kết thì khác: mỗi phần tử là một **nút** (node) riêng lẻ, chứa **giá trị** và **con trỏ** `next` chỉ tới nút kế tiếp. Nút cuối có `next = None`.

```text
head
 │
 ▼
┌───┬───┐   ┌───┬───┐   ┌───┬────┐
│ 1 │ ●─┼──►│ 2 │ ●─┼──►│ 3 │None│
└───┴───┘   └───┴───┘   └───┴────┘
 val next
```

Ta chỉ giữ con trỏ tới nút đầu (`head`). Muốn tới nút thứ `k` phải đi qua `k` bước — **không** có truy cập ngẫu nhiên.

| | Mảng | Danh sách liên kết |
|---|---|---|
| Lấy phần tử thứ `i` | `O(1)` | `O(i)` |
| Chèn/xóa ở đầu | `O(n)` | `O(1)` |
| Chèn/xóa sau một nút đã có trong tay | `O(n)` | `O(1)` |

In [1]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

# Tạo bằng tay: 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### Duyệt danh sách và hai hàm tiện ích

Mẫu duyệt cơ bản: con trỏ `cur` bắt đầu ở `head`, mỗi bước `cur = cur.next`, dừng khi `cur is None`.

In [2]:
def tao_danh_sach(values):
    """[1, 2, 3] -> 1 -> 2 -> 3, trả về head."""
    dummy = tail = ListNode()          # nút giả để khỏi xử lý riêng nút đầu
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def ra_list(head):
    out, cur = [], head
    while cur:                          # duyệt tới khi hết
        out.append(cur.val)
        cur = cur.next
    return out

h = tao_danh_sach([4, 5, 6, 7])
print("ra_list:", ra_list(h))
dem, cur = 0, h
while cur:
    dem += 1
    cur = cur.next
print("độ dài:", dem)

ra_list: [4, 5, 6, 7]
độ dài: 4


### Chèn và xóa: chỉ cần đổi con trỏ

Chèn nút `x` **sau** nút `p`: `x.next = p.next; p.next = x` (thứ tự quan trọng — làm ngược sẽ mất phần sau).

Xóa nút **sau** `p`: `p.next = p.next.next`.

```text
Chèn x sau p:   p ──► q        ⟹   p ──► x ──► q
Xóa sau p:      p ──► q ──► r  ⟹   p ──────────► r
```

Muốn xóa một nút, ta cần nút **đứng trước** nó. Nếu nút cần xóa là `head` thì không có nút trước → dùng **nút giả (dummy)** đặt trước `head`.

In [3]:
h = tao_danh_sach([1, 2, 4])
p = h.next                              # nút 2
x = ListNode(3)
x.next = p.next                         # 3 -> 4
p.next = x                              # 2 -> 3
print("sau khi chèn 3:", ra_list(h))

def xoa_gia_tri(head, target):
    dummy = ListNode(0, head)           # dummy đứng trước head
    prev = dummy
    while prev.next:
        if prev.next.val == target:
            prev.next = prev.next.next  # bỏ qua nút cần xóa
        else:
            prev = prev.next
    return dummy.next                   # head mới (có thể đã đổi)

print("xóa mọi số 1:", ra_list(xoa_gia_tri(tao_danh_sach([1, 1, 2, 1, 3]), 1)))

sau khi chèn 3: [1, 2, 3, 4]
xóa mọi số 1: [2, 3]


### Đảo ngược danh sách — kỹ thuật ba con trỏ

Đi dọc danh sách, lật mũi tên của từng nút cho trỏ về nút **trước** nó. Cần lưu `nxt` trước khi lật, vì sau khi lật ta mất đường đi tiếp.

```text
prev  cur   nxt
None   1 ──► 2 ──► 3          bước 1: 1.next = None
None ◄─1     2 ──► 3          bước 2: 2.next = 1
None ◄─1 ◄─ 2     3          bước 3: 3.next = 2  → head mới = 3
```

In [4]:
def dao_nguoc(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1) lưu đường đi tiếp
        cur.next = prev     # 2) lật mũi tên
        prev = cur          # 3) tiến prev
        cur = nxt           # 4) tiến cur
    return prev             # prev là head mới

print("đảo [1..5]:", ra_list(dao_nguoc(tao_danh_sach([1, 2, 3, 4, 5]))))

đảo [1..5]: [5, 4, 3, 2, 1]


### Con trỏ nhanh / chậm (fast & slow)

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng.

- Khi `fast` tới cuối, `slow` ở **giữa** danh sách.
- Nếu danh sách có **chu trình**, `fast` sẽ đuổi kịp `slow` (thuật toán Floyd).

In [5]:
def nut_giua(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("giữa [1..5]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5])).val,
      "| giữa [1..6]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5, 6])).val, "(nút giữa thứ hai)")

def co_chu_trinh(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

v = tao_danh_sach([1, 2, 3, 4])
v.next.next.next.next = v.next          # 4 -> 2 tạo chu trình
print("có chu trình?", co_chu_trinh(v), "| danh sách thường:", co_chu_trinh(tao_danh_sach([1, 2])))

giữa [1..5]: 3 | giữa [1..6]: 4 (nút giữa thứ hai)
có chu trình? True | danh sách thường: False


### Lỗi hay gặp với danh sách liên kết

- Truy cập `cur.next.val` khi `cur.next` là `None` → `AttributeError`. Luôn kiểm tra `cur and cur.next`.
- Đổi `cur.next` **trước** khi lưu nút kế → mất phần còn lại của danh sách.
- Quên trường hợp danh sách rỗng (`head is None`) hoặc chỉ một nút.
- Quên dùng dummy khi `head` có thể bị xóa/thay → phải viết nhánh riêng, dễ sai.
- In danh sách có chu trình bằng vòng `while cur` → lặp vô hạn.

## 2.2 Áp dụng vào bài này

Xóa **mọi** nút có giá trị bị lặp trên **danh sách liên kết** đã sắp xếp: dùng **nút giả** (vì `head` có thể bị xóa) và con trỏ `prev`; khi thấy một cụm giá trị bằng nhau thì bỏ qua cả cụm bằng `prev.next = nút sau cụm`.

# 3. Nền tảng: `ListNode` và các hàm tiện ích

In [6]:
from typing import Optional, List
import random


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values: List[int]) -> Optional[ListNode]:
    dummy = ListNode()
    tail = dummy
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head: Optional[ListNode]) -> List[int]:
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

print(to_list(build_list([1, 2, 3, 3, 4, 4, 5])))

[1, 2, 3, 3, 4, 4, 5]


## Vì sao cần nút giả (dummy)?

Ở ví dụ 2, ba nút đầu `1 -> 1 -> 1` đều bị xóa → **đầu danh sách thay đổi**. Nếu không có dummy, ta phải viết riêng logic "xóa ở đầu". Đặt `dummy.next = head`: mọi nút thật, kể cả nút đầu, đều có một nút **đứng trước**, nên xóa nút nào cũng như nhau. Cuối cùng trả `dummy.next`.

# 4. Baseline: đếm tần suất rồi dựng lại

Dễ hiểu, `O(n)` bộ nhớ phụ, và không tận dụng tính đã sắp xếp.

In [7]:
from collections import Counter

def delete_dup_counter(head: Optional[ListNode]) -> Optional[ListNode]:
    values = to_list(head)
    cnt = Counter(values)
    return build_list([v for v in values if cnt[v] == 1])

print(to_list(delete_dup_counter(build_list([1, 2, 3, 3, 4, 4, 5]))))
print(to_list(delete_dup_counter(build_list([1, 1, 1, 2, 3]))))

[1, 2, 5]
[2, 3]


# 5. Ý tưởng chính: `prev` + bỏ qua cả cụm

- `prev`: nút cuối cùng của phần kết quả (bắt đầu là `dummy`).
- `cur = prev.next`: nút đang xét.

Nếu `cur` và `cur.next` có **cùng giá trị** → đây là một cụm lặp: tiến `cur` tới **hết cụm**, rồi nối `prev.next` qua cả cụm (không di chuyển `prev`, vì nút tiếp theo có thể lại bắt đầu một cụm lặp khác).

Ngược lại → `cur` là giá trị duy nhất → giữ: `prev = cur`.

```text
dummy = ListNode(0, head); prev = dummy
while prev.next:
    cur = prev.next
    if cur.next and cur.next.val == cur.val:
        v = cur.val
        while cur and cur.val == v: cur = cur.next
        prev.next = cur            # bỏ cả cụm
    else:
        prev = cur                 # giữ cur
return dummy.next
```

In [8]:
def delete_dup(head: Optional[ListNode]) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next:
        cur = prev.next
        if cur.next and cur.next.val == cur.val:
            v = cur.val
            while cur and cur.val == v:
                cur = cur.next
            prev.next = cur
        else:
            prev = cur
    return dummy.next

print(to_list(delete_dup(build_list([1, 2, 3, 3, 4, 4, 5]))))
print(to_list(delete_dup(build_list([1, 1, 1, 2, 3]))))

[1, 2, 5]
[2, 3]


# 6. Mô phỏng

In [9]:
def trace_delete(values: List[int]) -> List[int]:
    head = build_list(values)
    dummy = ListNode("D", head)
    prev = dummy
    def show():
        out, node = [], dummy
        while node:
            out.append(str(node.val))
            node = node.next
        return " -> ".join(out)
    print("Ban đầu:", show())
    while prev.next:
        cur = prev.next
        if cur.next and cur.next.val == cur.val:
            v = cur.val
            while cur and cur.val == v:
                cur = cur.next
            prev.next = cur
            print(f"  cụm lặp {v!r:>3}: nối prev ({prev.val}) tới {cur.val if cur else None} | {show()}")
        else:
            prev = cur
            print(f"  giữ {cur.val!r:>3}: prev = {prev.val}")
    print("Kết quả:", to_list(dummy.next))
    return to_list(dummy.next)

_ = trace_delete([1, 2, 3, 3, 4, 4, 5])

Ban đầu: D -> 1 -> 2 -> 3 -> 3 -> 4 -> 4 -> 5
  giữ   1: prev = 1
  giữ   2: prev = 2
  cụm lặp   3: nối prev (2) tới 4 | D -> 1 -> 2 -> 4 -> 4 -> 5
  cụm lặp   4: nối prev (2) tới 5 | D -> 1 -> 2 -> 5
  giữ   5: prev = 5
Kết quả: [1, 2, 5]


Chú ý: sau khi bỏ cụm `3`, `prev` vẫn là `2`; vòng sau xét tiếp `4` — lại là một cụm lặp — và cũng bị bỏ. Nếu ta lỡ dời `prev` tới `4` thì `4` sẽ bị giữ sai.

# 7. Cách 2: Đệ quy

`f(head)` trả về danh sách đã xóa trùng bắt đầu từ `head`:

- Nếu `head` bắt đầu một cụm lặp → bỏ cả cụm, trả `f(nút sau cụm)`.
- Ngược lại → `head.next = f(head.next)`, trả `head`.

In [10]:
def delete_dup_rec(head: Optional[ListNode]) -> Optional[ListNode]:
    if not head or not head.next:
        return head
    if head.val == head.next.val:
        v = head.val
        while head and head.val == v:
            head = head.next
        return delete_dup_rec(head)
    head.next = delete_dup_rec(head.next)
    return head

print(to_list(delete_dup_rec(build_list([1, 2, 3, 3, 4, 4, 5]))))

[1, 2, 5]


Gọn, nhưng tốn `O(n)` ngăn xếp đệ quy (300 nút thì ổn; danh sách rất dài sẽ chạm giới hạn đệ quy của Python).

# 8. Phiên bản đúng format LeetCode

In [11]:
class Solution:
    def deleteDuplicates(self, head: Optional[ListNode]) -> Optional[ListNode]:
        dummy = ListNode(0, head)
        prev = dummy
        while prev.next:
            cur = prev.next
            if cur.next and cur.next.val == cur.val:
                v = cur.val
                while cur and cur.val == v:
                    cur = cur.next
                prev.next = cur
            else:
                prev = cur
        return dummy.next

In [12]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS  {actual}")


def run(values):
    return to_list(Solution().deleteDuplicates(build_list(values)))


check("Ví dụ 1", run([1, 2, 3, 3, 4, 4, 5]), [1, 2, 5])
check("Ví dụ 2", run([1, 1, 1, 2, 3]), [2, 3])
check("Rỗng", run([]), [])
check("Một nút", run([7]), [7])
check("Toàn trùng", run([2, 2, 2]), [])
check("Cụm ở cuối", run([1, 2, 2]), [1])
check("Hai cụm liền nhau", run([1, 1, 2, 2]), [])
check("Không trùng", run([1, 2, 3]), [1, 2, 3])

random.seed(82)
for _ in range(5000):
    values = sorted(random.randint(-3, 3) for _ in range(random.randint(0, 12)))
    expected = to_list(delete_dup_counter(build_list(values)))
    assert run(values) == expected == to_list(delete_dup_rec(build_list(values))), values
print("5000 danh sách ngẫu nhiên: khớp cách đếm tần suất và cách đệ quy")

Ví dụ 1                PASS  [1, 2, 5]


Ví dụ 2                PASS  [2, 3]
Rỗng                   PASS  []
Một nút                PASS  [7]
Toàn trùng             PASS  []
Cụm ở cuối             PASS  [1]
Hai cụm liền nhau      PASS  []
Không trùng            PASS  [1, 2, 3]


5000 danh sách ngẫu nhiên: khớp cách đếm tần suất và cách đệ quy


# 9. Vì sao đúng? Độ phức tạp?

**Bất biến:** danh sách từ `dummy` tới `prev` chỉ gồm các giá trị duy nhất (đúng kết quả cho phần đã xét), và `prev.next` là nút đầu tiên chưa xét.

- Vì danh sách đã sắp xếp, mọi bản của một giá trị nằm **liền nhau** → một cụm.
- `cur.next.val == cur.val` ⇔ `cur` mở đầu một cụm có ≥ 2 nút → bỏ cả cụm, `prev` giữ nguyên.
- Ngược lại `cur` là bản duy nhất → giữ, `prev` tiến lên.

Mỗi nút được `cur` đi qua đúng một lần.

```text
Time:  O(n)
Space: O(1) (lặp) | O(n) (đệ quy)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Giữ lại một bản (làm thành #83)

In [13]:
def wrong_keep_one(head: Optional[ListNode]) -> Optional[ListNode]:
    cur = head
    while cur and cur.next:
        if cur.val == cur.next.val:
            cur.next = cur.next.next              # SAI: còn giữ một bản
        else:
            cur = cur.next
    return head

print("Sai :", to_list(wrong_keep_one(build_list([1, 2, 3, 3, 4, 4, 5]))), "| Đúng:", run([1, 2, 3, 3, 4, 4, 5]))

Sai :

 [1, 2, 3, 4, 5] | Đúng: [1, 2, 5]


## Lỗi 2: Dời `prev` sau khi bỏ một cụm

In [14]:
def wrong_move_prev(head: Optional[ListNode]) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next:
        cur = prev.next
        if cur.next and cur.next.val == cur.val:
            v = cur.val
            while cur and cur.val == v:
                cur = cur.next
            prev.next = cur
            if cur:
                prev = cur                        # SAI: cur có thể mở đầu một cụm lặp mới
        else:
            prev = cur
    return dummy.next

print("Sai :", to_list(wrong_move_prev(build_list([1, 2, 3, 3, 4, 4, 5]))), "| Đúng:", run([1, 2, 3, 3, 4, 4, 5]))

Sai : [1, 2, 4, 4, 5] | Đúng: [1, 2, 5]


## Lỗi 3: Không dùng dummy, quên xử lý cụm ở đầu

In [15]:
def wrong_no_dummy(head: Optional[ListNode]) -> Optional[ListNode]:
    prev = head                                   # SAI: coi head luôn được giữ
    while prev and prev.next:
        cur = prev.next
        if cur.next and cur.next.val == cur.val:
            v = cur.val
            while cur and cur.val == v:
                cur = cur.next
            prev.next = cur
        else:
            prev = cur
    return head

print("Sai :", to_list(wrong_no_dummy(build_list([1, 1, 1, 2, 3]))), "| Đúng:", run([1, 1, 1, 2, 3]))

Sai : [1, 2, 3] | Đúng: [2, 3]


## Lỗi 4: Truy cập `cur.next.val` khi `cur.next` là `None`

Luôn kiểm tra `cur.next` trước (`if cur.next and cur.next.val == cur.val`), và `while cur and cur.val == v`.

# 11. Mẹo nhận diện pattern

**Nút giả (dummy head)** khi:

- Đầu danh sách có thể bị **xóa** hoặc **thay đổi**.
- Cần xây danh sách mới từng nút (`tail.next = ...`).

**Bỏ qua cả cụm** trên dữ liệu đã sắp xếp: `while cur and cur.val == v: cur = cur.next`.

Bài liên quan: **#83 Remove Duplicates from Sorted List**, **#203 Remove Linked List Elements**, **#19 Remove Nth Node From End of List**, **#1836 Remove Duplicates From an Unsorted Linked List**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Chạy thuật toán trên `1 -> 1 -> 2 -> 3 -> 3 -> 3 -> 4 -> 5 -> 5`. Kết quả?

## Bài 2 — Điền chỗ trống

In [16]:
def exercise_2_delete_duplicates(head: Optional[ListNode]) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next:
        cur = prev.next
        # TODO: nếu cur mở đầu cụm lặp -> bỏ cả cụm (prev giữ nguyên); ngược lại prev = cur
        pass
    return dummy.next


# assert to_list(exercise_2_delete_duplicates(build_list([1, 2, 3, 3, 4, 4, 5]))) == [1, 2, 5]
# assert to_list(exercise_2_delete_duplicates(build_list([1, 1, 1, 2, 3]))) == [2, 3]
# assert to_list(exercise_2_delete_duplicates(build_list([]))) == []

## Bài 3 — Biến thể: Remove Duplicates from Sorted List (LeetCode #83)

Giữ lại **một** bản của mỗi giá trị.

In [17]:
def exercise_3_delete_keep_one(head: Optional[ListNode]) -> Optional[ListNode]:
    # TODO
    pass


# assert to_list(exercise_3_delete_keep_one(build_list([1, 1, 2, 3, 3]))) == [1, 2, 3]
# assert to_list(exercise_3_delete_keep_one(build_list([]))) == []

## Bài 4 — Biến thể: danh sách **chưa** sắp xếp (LeetCode #1836)

Xóa mọi nút có giá trị xuất hiện nhiều hơn một lần, danh sách **không** sắp xếp.

```python
delete_unsorted(1 -> 2 -> 3 -> 2)  -> 1 -> 3
delete_unsorted(2 -> 1 -> 1 -> 2)  -> (rỗng)
```

Gợi ý nhanh: lượt 1 đếm tần suất; lượt 2 dùng dummy để xóa nút có tần suất > 1.

In [18]:
def exercise_4_delete_unsorted(head: Optional[ListNode]) -> Optional[ListNode]:
    # TODO
    pass


# assert to_list(exercise_4_delete_unsorted(build_list([1, 2, 3, 2]))) == [1, 3]
# assert to_list(exercise_4_delete_unsorted(build_list([2, 1, 1, 2]))) == []
# assert to_list(exercise_4_delete_unsorted(build_list([3, 2, 2, 1, 3, 2, 4]))) == [1, 4]

## Bài 5 — Mini challenge: giữ các giá trị xuất hiện **đúng `k` lần**

Danh sách đã sắp xếp. Chỉ giữ lại (đủ `k` bản) các giá trị xuất hiện **đúng** `k` lần; xóa các giá trị khác. Tại chỗ, `O(1)` bộ nhớ.

```python
keep_exact_k(1 -> 1 -> 2 -> 3 -> 3 -> 3, 2)  -> 1 -> 1
keep_exact_k(1 -> 2 -> 2 -> 3, 1)            -> 1 -> 3      # k = 1 chính là bài #82
```

In [19]:
def exercise_5_keep_exact_k(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    # TODO
    pass


# assert to_list(exercise_5_keep_exact_k(build_list([1, 1, 2, 3, 3, 3]), 2)) == [1, 1]
# assert to_list(exercise_5_keep_exact_k(build_list([1, 2, 2, 3]), 1)) == [1, 3]
# assert to_list(exercise_5_keep_exact_k(build_list([5, 5, 5]), 3)) == [5, 5, 5]

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_delete` sau khi tự làm.

**Bài 3**
- Mức 1: không cần dummy (nút đầu luôn được giữ). Nếu `cur.val == cur.next.val` thì `cur.next = cur.next.next`, ngược lại `cur = cur.next`.

**Bài 4**
- Mức 1: `Counter` trên các giá trị; rồi `prev = dummy`, nếu `cnt[prev.next.val] > 1` thì `prev.next = prev.next.next`, ngược lại `prev = prev.next`.

**Bài 5**
- Mức 1: với mỗi cụm bắt đầu tại `start = prev.next`, đếm độ dài `length` và tìm nút sau cụm `after`.
- Mức 2: nếu `length == k` thì giữ cả cụm: `prev` = nút cuối cụm; ngược lại `prev.next = after`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [20]:
_ = trace_delete([1, 1, 2, 3, 3, 3, 4, 5, 5])

Ban đầu: D -> 1 -> 1 -> 2 -> 3 -> 3 -> 3 -> 4 -> 5 -> 5
  cụm lặp   1: nối prev (D) tới 2 | D -> 2 -> 3 -> 3 -> 3 -> 4 -> 5 -> 5
  giữ   2: prev = 2
  cụm lặp   3: nối prev (2) tới 4 | D -> 2 -> 4 -> 5 -> 5
  giữ   4: prev = 4
  cụm lặp   5: nối prev (4) tới None | D -> 2 -> 4
Kết quả: [2, 4]


## Đáp án Bài 2

In [21]:
def answer_2_delete_duplicates(head: Optional[ListNode]) -> Optional[ListNode]:
    return delete_dup(head)

assert to_list(answer_2_delete_duplicates(build_list([1, 2, 3, 3, 4, 4, 5]))) == [1, 2, 5]
assert to_list(answer_2_delete_duplicates(build_list([1, 1, 1, 2, 3]))) == [2, 3]
assert to_list(answer_2_delete_duplicates(build_list([]))) == []
print("Bài 2 PASS (giống delete_dup ở mục 5)")

Bài 2 PASS (giống delete_dup ở mục 5)


## Đáp án Bài 3

In [22]:
def answer_3_delete_keep_one(head: Optional[ListNode]) -> Optional[ListNode]:
    cur = head
    while cur and cur.next:
        if cur.val == cur.next.val:
            cur.next = cur.next.next
        else:
            cur = cur.next
    return head

assert to_list(answer_3_delete_keep_one(build_list([1, 1, 2, 3, 3]))) == [1, 2, 3]
assert to_list(answer_3_delete_keep_one(build_list([]))) == []
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [23]:
def answer_4_delete_unsorted(head: Optional[ListNode]) -> Optional[ListNode]:
    cnt = Counter(to_list(head))
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next:
        if cnt[prev.next.val] > 1:
            prev.next = prev.next.next
        else:
            prev = prev.next
    return dummy.next

assert to_list(answer_4_delete_unsorted(build_list([1, 2, 3, 2]))) == [1, 3]
assert to_list(answer_4_delete_unsorted(build_list([2, 1, 1, 2]))) == []
assert to_list(answer_4_delete_unsorted(build_list([3, 2, 2, 1, 3, 2, 4]))) == [1, 4]
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [24]:
def answer_5_keep_exact_k(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next:
        start = prev.next
        last, length = start, 1
        while last.next and last.next.val == start.val:
            last = last.next
            length += 1
        if length == k:
            prev = last
        else:
            prev.next = last.next
    return dummy.next

assert to_list(answer_5_keep_exact_k(build_list([1, 1, 2, 3, 3, 3]), 2)) == [1, 1]
assert to_list(answer_5_keep_exact_k(build_list([1, 2, 2, 3]), 1)) == [1, 3]
assert to_list(answer_5_keep_exact_k(build_list([5, 5, 5]), 3)) == [5, 5, 5]
for _ in range(3000):
    k = random.randint(1, 3)
    values = sorted(random.randint(0, 4) for _ in range(random.randint(0, 12)))
    c = Counter(values)
    assert to_list(answer_5_keep_exact_k(build_list(values), k)) == [v for v in values if c[v] == k]
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Dummy + `prev`. Nếu `cur` mở đầu một cụm lặp (`cur.next.val == cur.val`), bỏ **cả cụm** bằng `prev.next = nút sau cụm` và **giữ nguyên** `prev`; ngược lại `prev = cur`. Trả `dummy.next`.

---

## Checklist trước khi nộp

- [ ] Có nút giả để xử lý cụm ở đầu?
- [ ] Không dời `prev` sau khi bỏ một cụm?
- [ ] Kiểm tra `None` trước khi truy cập `.val`?
- [ ] Trả `dummy.next`, không phải `head`?